# Lab 9- Deep Learning Model

This lab is meant to get you started in using Keras to design Deep Neural Networks. The goal here is to simply repeat your previous lab, but with DNNs.

Lecture 21 is your template: it reads the same data, builds, compiles and trains a network, plots its learning curves, and makes a ROC curve. Copy its cells and change the numbers.

Let's start with reading the data, like before. It reads the full file you downloaded in Lab 7 (5 million rows, about 760 MB in memory once loaded). If your computer struggles, read `../Lab.7/SUSY-small.csv` instead (the first 500,000 rows) and divide the sample sizes below by ten.

`RawNames` are the 8 quantities measured directly; `FeatureNames` are the 10 derived from them, as in Lab 8. You will need both in Exercise 2.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

filename="../Lab.7/SUSY.csv"
VarNames=["signal", "l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi", "MET", "MET_phi", "MET_rel", "axial_MET", "M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]
RawNames=["l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi", "MET", "MET_phi"]
FeatureNames=[v for v in VarNames[1:] if v not in RawNames]

df = pd.read_csv(filename, dtype='float64', names=VarNames)

Now let's define the samples. DNNs take long to train, so to start we use only the first 500,000 of the 5 million events for training. Once you get everything working, make the final version of your plots with more.

We keep **three** samples, because they do different jobs:

* **training**: what `fit` learns from;
* **validation**: what you watch while you work (pass it to `fit` as `validation_data`), and what you use to choose between models;
* **test**: used only at the end, to report the performance of the models you chose.

Keras works on NumPy arrays, so after doing all of the nice manipulation that pandas enables, we convert the DataFrames to NumPy arrays.

In [2]:
N_Train=500000
N_Val=50000
N_Test=50000

Train_Sample=df[:N_Train]
Val_Sample=df[N_Train:N_Train+N_Val]
Test_Sample=df[N_Train+N_Val:N_Train+N_Val+N_Test]

X_Train=np.array(Train_Sample[VarNames[1:]])
y_Train=np.array(Train_Sample["signal"])

X_Val=np.array(Val_Sample[VarNames[1:]])
y_Val=np.array(Val_Sample["signal"])

X_Test=np.array(Test_Sample[VarNames[1:]])
y_Test=np.array(Test_Sample["signal"])

print(X_Train.shape, X_Val.shape, X_Test.shape)

(500000, 18) (50000, 18) (50000, 18)


## Exercise 1

You will need to create several models and make sure they are properly trained. Write a function that takes this history and plots the values versus epoch. For every model that you train in the remainder of this lab, assess:

* Has your model's performance plateaued? If not train for more epochs. 
* Compare the performance on the training and validation samples. Are you over training?

In [3]:
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score

keras.utils.set_random_seed(42)

cols = VarNames[1:]
raw_idx  = [cols.index(v) for v in RawNames]
feat_idx = [cols.index(v) for v in FeatureNames]
all_idx  = list(range(len(cols)))
feature_sets_idx = {"Raw (8)": raw_idx, "Derived (10)": feat_idx, "Raw + Derived (18)": all_idx}


scaler = StandardScaler().fit(X_Train)
Xtr = scaler.transform(X_Train).astype("float32")
Xva = scaler.transform(X_Val).astype("float32")
Xte = scaler.transform(X_Test).astype("float32")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
def plot_history(history, title=""):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="validation")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title(title + " : loss"); ax[0].legend()
    ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="validation")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].set_title(title + " : accuracy"); ax[1].legend()
    plt.tight_layout(); plt.show()

def build_model(n_inputs, hidden=(64, 64), activation="relu",
                loss="binary_crossentropy", optimizer="adam", dropout=0.0):
    if callable(optimizer):            
        optimizer = optimizer()
    model = keras.Sequential()
    model.add(keras.Input(shape=(n_inputs,)))
    for n in hidden:
        model.add(layers.Dense(n, activation=activation))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(loss=loss, optimizer=optimizer, metrics=["accuracy"])
    return model

def train(model, idx, epochs=20, batch_size=512, title="", plot=True):
    history = model.fit(Xtr[:, idx], y_Train,
                        validation_data=(Xva[:, idx], y_Val),
                        epochs=epochs, batch_size=batch_size, verbose=1)
    if plot:
        plot_history(history, title)
    return history


m = build_model(len(all_idx))
h = train(m, all_idx, epochs=20, title="baseline")

## Exercise 2

Following the original paper (see Lab 7), make a comparison of the performance (using ROC curves and AUC) between models trained with the raw quantities (`RawNames`), the derived features (`FeatureNames`), and both. Note that the first layer's input shape changes with the number of inputs.

In [ ]:
results2 = {}
for name, idx in feature_sets_idx.items():
    model = build_model(len(idx), hidden=(64, 64, 64))
    train(model, idx, epochs=20, title=name)
    p_val  = model.predict(Xva[:, idx], verbose=0).ravel()
    p_test = model.predict(Xte[:, idx], verbose=0).ravel()
    results2[name] = dict(val_auc=roc_auc_score(y_Val, p_val),
                          test_auc=roc_auc_score(y_Test, p_test), p_test=p_test)

plt.figure(figsize=(7, 6))
for name, r in results2.items():
    fpr, tpr, _ = roc_curve(y_Test, r["p_test"])
    plt.plot(fpr, tpr, label=f"{name}, AUC = {r['test_auc']:.4f}")
plt.plot([0, 1], [0, 1], "k:", alpha=0.4)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("DNN: raw vs. derived vs. all (test sample)"); plt.legend(loc="lower right"); plt.show()

for name, r in results2.items():
    print(f"{name:22s} val AUC = {r['val_auc']:.4f}   test AUC = {r['test_auc']:.4f}")

## Exercise 3

Design and implement at least 3 different DNN models. Train them and compare performance. You may try different architectures, loss functions, and optimizers to see if there is an effect.

In [ ]:
configs = {
    "A: 3x64 relu, adam, BCE":     dict(hidden=(64, 64, 64), activation="relu", optimizer="adam"),
    "B: 4x128 relu, dropout 0.1":  dict(hidden=(128, 128, 128, 128), activation="relu", optimizer="adam", dropout=0.1),
    "C: 3x64 tanh, SGD+momentum":  dict(hidden=(64, 64, 64), activation="tanh",
                                        optimizer=lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)),
    "D: 3x64 relu, adam, MSE":     dict(hidden=(64, 64, 64), activation="relu", optimizer="adam",
                                        loss="mean_squared_error"),
}

results3 = {}
for name, cfg in configs.items():
    model = build_model(len(all_idx), **cfg)
    train(model, all_idx, epochs=25, title=name)
    p_val  = model.predict(Xva, verbose=0).ravel()
    p_test = model.predict(Xte, verbose=0).ravel()
    results3[name] = dict(val_auc=roc_auc_score(y_Val, p_val),
                          test_auc=roc_auc_score(y_Test, p_test), p_test=p_test)


for name, r in results3.items():
    print(f"{name:30s} val AUC = {r['val_auc']:.4f}")
best_name = max(results3, key=lambda k: results3[k]["val_auc"])
print("\nBest DNN (by validation AUC):", best_name)


plt.figure(figsize=(7, 6))
for name, r in results3.items():
    fpr, tpr, _ = roc_curve(y_Test, r["p_test"])
    plt.plot(fpr, tpr, label=f"{name}, AUC = {r['test_auc']:.4f}")
plt.plot([0, 1], [0, 1], "k:", alpha=0.4)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("DNN architectures (test sample)"); plt.legend(loc="lower right", fontsize=8); plt.show()

## Exercise 4

Repeat Exercise 4 from Lab 8 (parts b and c), adding your best performing DNN as one of the models. Choose your best DNN using the validation sample, then compare on the test sample.

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
import sklearn.discriminant_analysis as DA
from sklearn.metrics import auc

def score(clf, X):
    return clf.decision_function(X) if hasattr(clf, "decision_function") else clf.predict_proba(X)[:, 1]

def sk_method(clf):
    def run(idx):
        c = clone(clf).fit(X_Train[:, idx], y_Train)
        return score(c, X_Test[:, idx])
    return run

def dnn_method(cfg, epochs=25):
    def run(idx):
        sc_ = StandardScaler().fit(X_Train[:, idx])
        m = build_model(len(idx), **cfg)
        m.fit(sc_.transform(X_Train[:, idx]).astype("float32"), y_Train,
              validation_data=(sc_.transform(X_Val[:, idx]).astype("float32"), y_Val),
              epochs=epochs, batch_size=512, verbose=0)
        return m.predict(sc_.transform(X_Test[:, idx]).astype("float32"), verbose=0).ravel()
    return run

methods = {
    "Logistic Regression":     sk_method(make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))),
    "Quadratic Discriminant":  sk_method(DA.QuadraticDiscriminantAnalysis()),
    "Hist. Gradient Boosting": sk_method(HistGradientBoostingClassifier(max_iter=200)),
    f"Best DNN ({best_name.split(':')[0]})": dnn_method(configs[best_name]),
}

def compare_feature_sets(run, label=""):
    plt.figure(figsize=(7, 6))
    aucs, scores = {}, {}
    for name, idx in feature_sets_idx.items():
        s = run(idx)
        fpr, tpr, _ = roc_curve(y_Test, s)
        aucs[name], scores[name] = auc(fpr, tpr), s
        plt.plot(fpr, tpr, label=f"{name}, AUC = {aucs[name]:.4f}")
    plt.plot([0, 1], [0, 1], "k:", alpha=0.4)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"{label} (test sample)"); plt.legend(loc="lower right"); plt.show()
    return aucs, scores

all_aucs, all_scores = {}, {}
for name, run in methods.items():
    all_aucs[name], all_scores[name] = compare_feature_sets(run, label=name)

display(HTML(tabulate.tabulate(
    [[m] + [f"{a[k]:.4f}" for k in feature_sets_idx] for m, a in all_aucs.items()],
    headers=["method"] + list(feature_sets_idx), tablefmt="html")))

In [ ]:
Scenarios = {1:(10,100), 2:(100,1000), 3:(1000,10000), 4:(10000,100000)}

def signif(eS, eB, NS, NB):
    NSp, NBp = eS*NS, eB*NB
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(NSp+NBp > 0, NSp/np.sqrt(NSp+NBp), 0.0)

best_method = max(all_aucs, key=lambda k: all_aucs[k]["Raw + Derived (18)"])
print("Best method (by AUC, all 18 inputs):", best_method)
s_test = all_scores[best_method]["Raw + Derived (18)"]

fpr, tpr, thr = roc_curve(y_Test, s_test)
ok = fpr > 1e-3

best_cut = {}
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (sc, (NS, NB)) in zip(axes, Scenarios.items()):
    sg = np.where(ok, signif(tpr, fpr, NS, NB), 0)
    k = np.argmax(sg)
    best_cut[sc] = dict(threshold=thr[k], significance=sg[k], tpr=tpr[k], fpr=fpr[k])
    ax.plot(thr[1:], sg[1:], "k"); ax.axvline(thr[k], color="g", ls="--")
    ax.set_xlabel("cut on classifier output"); ax.set_ylabel("$\\sigma_S$")
    ax.set_title(f"Scenario {sc}: N_S={NS}, N_B={NB}", fontsize=10)
    print(f"Scenario {sc}: max sigma_S = {sg[k]:.3f} at cut {thr[k]:.4f} (TPR={tpr[k]:.3f}, FPR={fpr[k]:.4f})")
fig.tight_layout(); plt.show()

In [ ]:
Lab 9

Exercise 1: 

Plateau: if validation loss and accuracy are flat over the last several epochs, it has plateaued. If they are still improving, train for more epochs.
Overtraining: if training loss keeps falling while validation loss stops falling or rises, with a growing gap, it is overtraining. On SUSY with 500k training events, expect a small gap for the 3×64 networks and a larger one for the 4×128 network. Dropout should shrink the gap.

Exercise 2: 
This is the main result of the original paper. The DNN trained on raw inputs alone should perform nearly as well as, or about as well as, the DNN trained on derived features, and combining them should add little. The network learns the derived quantities itself. The paper reports AUCs in the high 0.8s for SUSY, with the DNN on raw inputs and on all inputs very close together.

Exercise 3:
Configurations A, B and D (relu networks with Adam) should land within a thousandth or two of each other, since SUSY separation is limited by the physics more than the architecture. The tanh/SGD model (C) will probably train more slowly and end slightly lower in the same number of epochs. The MSE-loss model (D) usually trains a bit worse than cross-entropy. Choose the winner by validation AUC, then quote its test AUC.

Exercise 4: 
The DNN should match or slightly beat gradient boosting on all 18 inputs and clearly beat the linear and quadratic methods on the raw inputs. With only 500k training events and a 50k test set, differences of a few thousandths in AUC are within noise, so say that if your numbers are that close. For the significance, the smallest scenarios (N_S=10) will be noisy with a 50k test set. That is a reason to enlarge N_Test for the final version.